In [ ]:
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment

from head_detector import HeadDetector


def load_dlib_landmarks(label_path: str) -> np.ndarray:
    """
    Load DLIB-style 68 landmarks from a text file.

    The expected format is one coordinate pair per line:
        x1 y1
        ...
        x68 y68

    Args:
        label_path: Path to the landmark label file.

    Returns:
        A NumPy array with shape (68, 2).

    Raises:
        FileNotFoundError: If the label file does not exist.
        ValueError: If the file does not contain exactly 68 coordinate pairs.
    """
    label_path = Path(label_path)

    if not label_path.exists():
        raise FileNotFoundError(f"Label file not found: {label_path}")

    landmarks = np.loadtxt(label_path, dtype=np.float32)

    if landmarks.ndim == 1:
        landmarks = landmarks.reshape(-1, 2)

    if landmarks.shape != (68, 2):
        raise ValueError(
            f"Expected landmarks with shape (68, 2), got {landmarks.shape} from {label_path}"
        )

    return landmarks


def select_best_head(predictions) -> object:
    """
    Select the highest-confidence detected head from VGGHeads predictions.

    Args:
        predictions: PredictionResult returned by HeadDetector.

    Returns:
        The highest-confidence HeadMetadata object.

    Raises:
        ValueError: If no heads are detected.
    """
    if not predictions.heads:
        raise ValueError("No heads were detected.")

    return max(predictions.heads, key=lambda head: float(head.score))


def extract_vertices_2d(head) -> np.ndarray:
    """
    Extract 2D projected VGGHeads mesh vertices.

    Args:
        head: HeadMetadata object returned by VGGHeads.

    Returns:
        A NumPy array with shape (N, 2) containing projected mesh vertices
        in original image coordinates.
    """
    return head.vertices_3d[:, :2].astype(np.float32)


def draw_points(
    image_rgb: np.ndarray,
    points: np.ndarray,
    color_rgb: Tuple[int, int, int] = (0, 255, 255),
    radius: int = 3,
) -> np.ndarray:
    """
    Draw 2D points on an RGB image.

    Args:
        image_rgb: Input RGB image.
        points: Array with shape (N, 2).
        color_rgb: Point color in RGB format.
        radius: Circle radius.

    Returns:
        RGB image with drawn points.
    """
    output_image = image_rgb.copy()

    for x_coord, y_coord in points:
        cv2.circle(
            output_image,
            (int(round(x_coord)), int(round(y_coord))),
            radius,
            color_rgb,
            -1,
        )

    return output_image


def save_landmarks_txt(landmarks: np.ndarray, output_path: str | Path) -> None:
    """
    Save 2D landmarks to a text file.

    Args:
        landmarks: Array with shape (N, 2).
        output_path: Destination text file.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    np.savetxt(output_path, landmarks, fmt="%.6f")


def run_vggheads_single_image(
    image_path: str,
    output_image_path: str,
    output_txt_path: str,
    confidence_threshold: float = 0.5,
) -> np.ndarray:
    """
    Run VGGHeads inference on a single image and save projected mesh vertices.

    Args:
        image_path: Path to the input image.
        output_image_path: Path where the visualization image will be saved.
        output_txt_path: Path where the projected vertices will be saved.
        confidence_threshold: Minimum detection confidence.

    Returns:
        Array with shape (N, 2) containing projected VGGHeads mesh vertices.
    """
    detector = HeadDetector()
    predictions = detector(str(image_path), confidence_threshold=confidence_threshold)

    best_head = select_best_head(predictions)
    vertices_2d = extract_vertices_2d(best_head)

    output_image_path = Path(output_image_path)
    output_image_path.parent.mkdir(parents=True, exist_ok=True)

    image_with_points = draw_points(
        image_rgb=predictions.original_image,
        points=vertices_2d,
        color_rgb=(0, 255, 255),
        radius=2,
    )

    cv2.imwrite(
        str(output_image_path),
        cv2.cvtColor(image_with_points, cv2.COLOR_RGB2BGR),
    )

    save_landmarks_txt(vertices_2d, output_txt_path)

    return vertices_2d


def compute_distance_matrix(
    dlib_landmarks: np.ndarray,
    vgg_vertices_2d: np.ndarray,
) -> np.ndarray:
    """
    Compute pairwise Euclidean distances between DLIB landmarks and VGGHeads vertices.

    Args:
        dlib_landmarks: Array with shape (68, 2).
        vgg_vertices_2d: Array with shape (N, 2).

    Returns:
        Distance matrix with shape (68, N).
    """
    diff = dlib_landmarks[:, None, :] - vgg_vertices_2d[None, :, :]
    return np.linalg.norm(diff, axis=2)


def match_dlib_to_vggheads_indices(
    dlib_landmarks: np.ndarray,
    vgg_vertices_2d: np.ndarray,
    max_distance_px: float = None,
) -> List[int]:
    """
    Match DLIB 68 landmarks to VGGHeads mesh vertices using global assignment.

    This is more robust than greedy nearest-neighbor matching because it minimizes
    the total matching cost while enforcing unique VGGHeads indices.

    Args:
        dlib_landmarks: Array with shape (68, 2).
        vgg_vertices_2d: Array with shape (N, 2).
        max_distance_px: Optional maximum allowed distance in pixels. If provided,
            an error is raised when any matched point exceeds this threshold.

    Returns:
        A list of 68 VGGHeads vertex indices ordered according to the DLIB template.

    Raises:
        ValueError: If input shapes are invalid or if a match exceeds max_distance_px.
    """
    if dlib_landmarks.shape != (68, 2):
        raise ValueError(f"Expected dlib_landmarks shape (68, 2), got {dlib_landmarks.shape}")

    if vgg_vertices_2d.ndim != 2 or vgg_vertices_2d.shape[1] != 2:
        raise ValueError(f"Expected vgg_vertices_2d shape (N, 2), got {vgg_vertices_2d.shape}")

    distance_matrix = compute_distance_matrix(dlib_landmarks, vgg_vertices_2d)
    row_indices, column_indices = linear_sum_assignment(distance_matrix)

    if not np.array_equal(row_indices, np.arange(68)):
        raise RuntimeError("Unexpected assignment order. DLIB landmark rows were not preserved.")

    matched_distances = distance_matrix[row_indices, column_indices]

    if max_distance_px is not None and np.any(matched_distances > max_distance_px):
        worst_idx = int(np.argmax(matched_distances))
        raise ValueError(
            f"Match distance too large for DLIB landmark {worst_idx}: "
            f"{matched_distances[worst_idx]:.2f}px > {max_distance_px:.2f}px"
        )

    return column_indices.astype(int).tolist()


def save_mapping(
    vgg_indices_68: List[int],
    output_json_path: str,
) -> None:
    """
    Save the DLIB-to-VGGHeads vertex mapping to JSON.

    Args:
        vgg_indices_68: List of 68 VGGHeads vertex indices.
        output_json_path: Destination JSON file.
    """
    output_json_path = Path(output_json_path)
    output_json_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "description": "Static mapping from DLIB 68 landmarks to VGGHeads mesh vertex indices.",
        "vgg_indices_68": vgg_indices_68,
    }

    with open(output_json_path, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2)


def load_mapping(mapping_json_path: str | Path) -> List[int]:
    """
    Load a saved DLIB-to-VGGHeads vertex mapping.

    Args:
        mapping_json_path: Path to the mapping JSON file.

    Returns:
        List of 68 VGGHeads vertex indices.
    """
    with open(mapping_json_path, "r", encoding="utf-8") as file:
        payload = json.load(file)

    indices = payload["vgg_indices_68"]

    if len(indices) != 68:
        raise ValueError(f"Expected 68 indices, got {len(indices)}")

    return [int(index) for index in indices]


def extract_vggheads_68_from_mapping(
    vgg_vertices_2d: np.ndarray,
    vgg_indices_68: List[int],
) -> np.ndarray:
    """
    Extract 68 DLIB-style landmarks from VGGHeads projected mesh vertices.

    Args:
        vgg_vertices_2d: Array with shape (N, 2).
        vgg_indices_68: Static list of 68 VGGHeads vertex indices.

    Returns:
        Array with shape (68, 2).
    """
    indices = np.asarray(vgg_indices_68, dtype=np.int64)
    return vgg_vertices_2d[indices]


def build_mapping_from_single_reference(
    image_path: str,
    dlib_label_path: str,
    output_mapping_json_path: str,
    output_vgg_vertices_txt_path: str,
    output_visualization_path: str,
    max_distance_px: float = None,
) -> List[int]:
    """
    Build a static DLIB 68 to VGGHeads vertex mapping from one reference image.

    Args:
        image_path: Path to the reference image.
        dlib_label_path: Path to the DLIB 68 label file.
        output_mapping_json_path: Path where the mapping JSON will be saved.
        output_vgg_vertices_txt_path: Path where all VGGHeads vertices will be saved.
        output_visualization_path: Path where the VGGHeads visualization will be saved.
        max_distance_px: Optional maximum allowed matching distance.

    Returns:
        List of 68 VGGHeads vertex indices.
    """
    dlib_landmarks = load_dlib_landmarks(dlib_label_path)

    vgg_vertices_2d = run_vggheads_single_image(
        image_path=image_path,
        output_image_path=output_visualization_path,
        output_txt_path=output_vgg_vertices_txt_path,
    )

    vgg_indices_68 = match_dlib_to_vggheads_indices(
        dlib_landmarks=dlib_landmarks,
        vgg_vertices_2d=vgg_vertices_2d,
        max_distance_px=max_distance_px,
    )

    save_mapping(vgg_indices_68, output_mapping_json_path)

    return vgg_indices_68

In [ ]:
image_path = "/home/jocareher/Documents/baby_face_72/images/face_bcn_02.JPG"
dlib_label_path = "/home/jocareher/Documents/baby_face_72/labels/face_bcn_02.txt"

mapping = build_mapping_from_single_reference(
    image_path=image_path,
    dlib_label_path=dlib_label_path,
    output_mapping_json_path="outputs/vgg_to_dlib68_mapping.json",
    output_vgg_vertices_txt_path="outputs/face_bcn_02_vgg_vertices.txt",
    output_visualization_path="outputs/face_bcn_02_vgg_vertices.png",
    max_distance_px=25.0,
)

print(mapping)

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
from head_detector import HeadDetector


def load_dlib_landmarks(label_path: str | Path) -> np.ndarray:
    """
    Load DLIB 68 landmarks from a text file.

    Args:
        label_path: Path to a txt file with one coordinate pair per line.

    Returns:
        Array with shape (68, 2).
    """
    landmarks = np.loadtxt(label_path, dtype=np.float32)

    if landmarks.ndim == 1:
        landmarks = landmarks.reshape(-1, 2)

    if landmarks.shape != (68, 2):
        raise ValueError(f"Expected shape (68, 2), got {landmarks.shape}: {label_path}")

    return landmarks


def select_best_head(predictions) -> object:
    """
    Select the highest-confidence detected head.

    Args:
        predictions: VGGHeads PredictionResult.

    Returns:
        Highest-confidence head.
    """
    if not predictions.heads:
        raise ValueError("No heads detected.")

    return max(predictions.heads, key=lambda head: float(head.score))


def run_vggheads_on_image(
    detector: HeadDetector,
    image_path: str | Path,
    confidence_threshold: float = 0.5,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Run VGGHeads on one image and return projected vertices.

    Args:
        detector: Initialized HeadDetector.
        image_path: Input image path.
        confidence_threshold: Detection threshold.

    Returns:
        Tuple containing:
            - original RGB image
            - projected vertices with shape (N, 2)
    """
    predictions = detector(str(image_path), confidence_threshold=confidence_threshold)
    best_head = select_best_head(predictions)
    vertices_2d = best_head.vertices_3d[:, :2].astype(np.float32)

    return predictions.original_image, vertices_2d


def draw_points(
    image_rgb: np.ndarray,
    points: np.ndarray,
    color_rgb: Tuple[int, int, int],
    radius: int = 3,
) -> np.ndarray:
    """
    Draw points on an RGB image.

    Args:
        image_rgb: RGB image.
        points: Points with shape (N, 2).
        color_rgb: RGB color.
        radius: Point radius.

    Returns:
        RGB image with points.
    """
    output = image_rgb.copy()

    for x_coord, y_coord in points:
        cv2.circle(
            output,
            (int(round(x_coord)), int(round(y_coord))),
            radius,
            color_rgb,
            -1,
        )

    return output


def process_reference_set_with_vggheads(
    image_dir: str | Path,
    dlib_label_dir: str | Path,
    output_dir: str | Path,
    confidence_threshold: float = 0.5,
) -> None:
    """
    Run VGGHeads on a reference set with existing DLIB labels.

    Args:
        image_dir: Directory containing input images.
        dlib_label_dir: Directory containing DLIB 68 txt labels.
        output_dir: Directory where VGGHeads outputs will be saved.
        confidence_threshold: Detection threshold.
    """
    image_dir = Path(image_dir)
    dlib_label_dir = Path(dlib_label_dir)
    output_dir = Path(output_dir)

    vertices_dir = output_dir / "vertices"
    visualization_dir = output_dir / "visualizations"

    vertices_dir.mkdir(parents=True, exist_ok=True)
    visualization_dir.mkdir(parents=True, exist_ok=True)

    detector = HeadDetector()

    image_paths = sorted(
        path for path in image_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )

    for image_path in image_paths:
        label_path = dlib_label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            print(f"[WARNING] Missing DLIB label: {label_path}")
            continue

        dlib_landmarks = load_dlib_landmarks(label_path)
        image_rgb, vertices_2d = run_vggheads_on_image(
            detector=detector,
            image_path=image_path,
            confidence_threshold=confidence_threshold,
        )

        np.savetxt(vertices_dir / f"{image_path.stem}.txt", vertices_2d, fmt="%.6f")

        image_with_dlib = draw_points(
            image_rgb=image_rgb,
            points=dlib_landmarks,
            color_rgb=(255, 0, 0),
            radius=3,
        )

        image_with_both = draw_points(
            image_rgb=image_with_dlib,
            points=vertices_2d,
            color_rgb=(0, 255, 255),
            radius=1,
        )

        cv2.imwrite(
            str(visualization_dir / f"{image_path.stem}.png"),
            cv2.cvtColor(image_with_both, cv2.COLOR_RGB2BGR),
        )

        print(f"[OK] Processed {image_path.name}")

In [ ]:
process_reference_set_with_vggheads(
    image_dir="/path/to/reference_set/images",
    dlib_label_dir="/path/to/reference_set/dlib_labels",
    output_dir="/path/to/reference_set/vgg_outputs",
    confidence_threshold=0.5,
)